# Deep Learning Master Class — Colab Uygulaması
## Yüz Analizi: bir fotoğraftan yaş, cinsiyet ve duygu

**Kuantum Bootcamp · Veysel Murat Görken**

---

Bugün konuştuğumuz her şey burada bir araya geliyor:

| Sunumda gördüğümüz | Bu defterde |
|---|---|
| CNN — filtreler görüntüde gezer | Yüzü bulan model bir CNN |
| Softmax — olasılık dağılımı | Duygu tahmini 7 sınıfa dağıtılmış olasılık |
| Embedding — anlam bir vektöre dönüşür | Yüz de 512 boyutlu bir vektöre dönüşüyor |
| Ölçekleme | Görüntü 224x224'e getirilip normalize ediliyor |

Hiçbir şey eğitmiyoruz. Eğitilmiş modelleri kullanıyoruz — bugün sahadaki
işlerin çoğu böyle başlıyor.

> Bu defter, [generative-ai-workshop](https://github.com/grknc/generative-ai-workshop)
> projesindeki **Yüz Analizi** sekmesinin Python karşılığıdır. Orada modeller
> tarayıcıda (face-api.js) çalışıyor; burada Colab'da, `DeepFace` ile.


---
## 0 · Kurulum

`deepface`, yüz tespiti ve analizi için hazır modelleri tek satırla sunan bir
kütüphane. Kurulum yaklaşık 1–2 dakika sürüyor.

> **Ne oluyor:** OpenCV 5.0 ile `CascadeClassifier` çekirdekten kaldırıldı,
> `deepface` ise hâlâ 4.x API'sini kullanıyor. `pip` varsayılan olarak en yeniyi
> çektiği için `cv2` yarım görünüyor. Aşağıdaki hücre OpenCV'yi **4.x'e
> sabitliyor** ve ardından oturumu kendiliğinden yeniden başlatıyor.
>
> "Oturum yeniden başlatıldı" uyarısını gördükten sonra bu hücreyi **atlayıp**
> bir alttakinden devam edin.


In [ ]:
import os, sys, subprocess

ISARET = "/content/.kurulum_tamam_v3"
OPENCV = "opencv-python-headless>=4.10,<5"     # 5.0 CascadeClassifier'ı kaldırdı


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "-q", *args], check=False)


if os.path.exists(ISARET):
    print("Kurulum zaten yapılmış — atlanıyor.")
else:
    pip("install", "deepface", "tf-keras")
    # deepface, opencv-python'ı geri çekiyor; en sonda tek ve 4.x sürüm bırakıyoruz
    pip("uninstall", "-y", "opencv-python", "opencv-contrib-python",
        "opencv-python-headless", "opencv-contrib-python-headless")
    pip("install", OPENCV)
    open(ISARET, "w").close()

    print("Kurulum bitti. Oturum yeniden başlatılıyor (birkaç saniye)...")
    print("Sonra BU HÜCREYİ ATLAYIP bir alttaki hücreden devam edin.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)


<details>
<summary><b>Teşhis</b> — <code>cv2</code> hatası devam ederse</summary>

Kurulum hücresini yeniden başlatmadan çalıştırdıysanız hata sürer.
Elle çözüm: **Çalışma zamanı → Oturumu yeniden başlat**, sonra kurulum
hücresini atlayıp devam edin. Yine olmazsa aşağıdaki hücrenin çıktısına bakın.

</details>


In [ ]:
# Sorun sürerse bu hücreyi çalıştırıp çıktısını paylaşın.
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pip", "list"],
                     capture_output=True, text=True).stdout
      .replace("\r", "").split("\n")[0])
for satir in subprocess.run([sys.executable, "-m", "pip", "list"],
                            capture_output=True, text=True).stdout.split("\n"):
    if "opencv" in satir.lower():
        print(satir)

import cv2
print("cv2 sürüm :", cv2.__version__)
print("cv2 dosya :", cv2.__file__)
print("CascadeClassifier var mı :", hasattr(cv2, "CascadeClassifier"))


In [ ]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import cv2
if not hasattr(cv2, "CascadeClassifier"):
    raise RuntimeError(
        f"OpenCV {cv2.__version__} uygun değil (CascadeClassifier yok). "
        "Çalışma zamanı → Çalışma zamanını sil, sonra defteri baştan çalıştırın; "
        "kurulum hücresi OpenCV'yi 4.x'e sabitliyor."
    )

import requests, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image
from deepface import DeepFace

print("OpenCV :", cv2.__version__)
print("Hazır.")


---
## 1 · Bir yüz bulalım

Demo için **yapay üretilmiş** bir yüz kullanıyoruz. Aşağıdaki fotoğraftaki kişi
gerçekte yok — bir GAN tarafından üretildi.

> **Konuşma notu:** Sunumdaki 'üretken vs klasik' slaydını hatırlayın. Bu fotoğrafı
> bir **üretken** model üretti; şimdi onu bir **ayırt edici** modele vereceğiz.
> İki dünya aynı hücrede karşılaşıyor.

Bağlantı çalışmazsa 4. bölümdeki yükleme hücresini kullanın.


In [ ]:
URET_URL = "https://thispersondoesnotexist.com/random-person.jpeg"


def yuz_indir(url=URET_URL):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (DL MasterClass)"},
                     timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


try:
    resim = yuz_indir()
except Exception as e:
    print("İndirilemedi:", e, "\n→ 4. bölümden kendi fotoğrafınızı yükleyin.")
    resim = None

if resim is not None:
    resim.save("foto.jpg")
    plt.figure(figsize=(5, 5))
    plt.imshow(resim); plt.axis("off")
    plt.title(f"Girdi — {resim.size[0]}x{resim.size[1]} piksel")
    plt.show()


---
## 2 · Adım 1 — Yüzü bul (face detection)

Analiz yapmadan önce fotoğrafta yüzün **nerede** olduğunu bulmak gerekiyor.
Bunu yapan şey bir CNN: küçük filtreler görüntünün üzerinde geziyor ve
"burada yüz var" diyor.

Bulunan bölge kırpılıp 224x224'e ölçekleniyor — sonraki modellerin beklediği boyut.


In [ ]:
DETEKTOR = "opencv"      # hızlı; sorun çıkarsa otomatik olarak retinaface'e geçer


def yuzleri_bul(dosya, backend=None):
    global DETEKTOR
    try:
        return DeepFace.extract_faces(img_path=dosya,
                                      detector_backend=backend or DETEKTOR,
                                      enforce_detection=False)
    except Exception as e:
        print(f"{DETEKTOR} çalışmadı ({type(e).__name__}) → retinaface deneniyor.")
        DETEKTOR = "retinaface"
        return DeepFace.extract_faces(img_path=dosya,
                                      detector_backend=DETEKTOR,
                                      enforce_detection=False)


yuzler = yuzleri_bul("foto.jpg")
print(f"Bulunan yüz sayısı: {len(yuzler)}")

fig, eksen = plt.subplots(1, 2, figsize=(11, 5))
eksen[0].imshow(resim)
for y in yuzler:
    a = y["facial_area"]
    eksen[0].add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"],
                                         fill=False, edgecolor="#C08A2E", lw=3))
eksen[0].set_title("Tespit edilen yüz"); eksen[0].axis("off")

eksen[1].imshow(yuzler[0]["face"])
eksen[1].set_title("Kırpılmış ve ölçeklenmiş — modele giren görüntü")
eksen[1].axis("off")
plt.tight_layout(); plt.show()

kirpik = yuzler[0]["face"]
print("Modele giren dizinin boyutu :", kirpik.shape)
print(f"Değer aralığı               : {kirpik.min():.2f} ... {kirpik.max():.2f}")


> **Konuşma notu:** Sağdaki görüntü artık bir fotoğraf değil, `224 x 224 x 3` boyutunda bir sayı kümesi — yani 150.528 sayı. Ev fiyatı örneğinde 4 sayı vardı; fark sadece bu.


---
## 3 · Adım 2 — Duygu analizi (softmax çalışıyor)

Duygu modeli 7 sınıf üretiyor: kızgın, tiksinti, korku, mutlu, üzgün, şaşkın, nötr.
Çıkışta **softmax** var — tam olarak sunumda konuştuğumuz gibi, olasılıkların
toplamı 1 oluyor.


In [ ]:
DUYGU_TR = {
    "angry": "kızgın", "disgust": "tiksinti", "fear": "korku",
    "happy": "mutlu", "sad": "üzgün", "surprise": "şaşkın", "neutral": "nötr",
}
CINSIYET_TR = {"Man": "Erkek", "Woman": "Kadın"}


def analiz(dosya, eylemler):
    return DeepFace.analyze(
        img_path=dosya,
        actions=eylemler,
        detector_backend=DETEKTOR,
        enforce_detection=False,
    )[0]


duygu = analiz("foto.jpg", ["emotion"])

sirali = sorted(duygu["emotion"].items(), key=lambda x: -x[1])
print("Baskın duygu:", DUYGU_TR.get(duygu["dominant_emotion"], duygu["dominant_emotion"]).upper())
print("-" * 34)
for ad, p in sirali:
    print(f"  %{p:5.1f}   {DUYGU_TR.get(ad, ad)}")

adlar = [DUYGU_TR.get(a, a) for a, _ in sirali][::-1]
degerler = [p for _, p in sirali][::-1]
plt.figure(figsize=(7, 3.2))
plt.barh(adlar, degerler, color="#148F77")
plt.xlabel("olasılık (%)"); plt.title("Duygu dağılımı (softmax çıktısı)")
plt.xlim(0, 100)
plt.tight_layout(); plt.show()

print(f"\nOlasılıkların toplamı: {sum(duygu['emotion'].values()):.1f}")


---
## 4 · Adım 3 — Yaş ve cinsiyet

Aynı yüz, iki farklı modele daha giriyor.

- **Yaş** — regresyon problemi: çıkış tek bir sayı (ev fiyatı örneğindeki gibi)
- **Cinsiyet** — ikili sınıflandırma: iki sınıfa dağıtılmış olasılık

> İlk çalıştırmada model ağırlıkları indirilir, biraz sürebilir.


In [ ]:
hepsi = analiz("foto.jpg", ["age", "gender", "emotion"])

print("=" * 40)
print(f"  Yaş tahmini    : {hepsi['age']}")
g = hepsi["dominant_gender"]
print(f"  Cinsiyet       : {CINSIYET_TR.get(g, g)}  (%{hepsi['gender'][g]:.1f})")
d = hepsi["dominant_emotion"]
print(f"  Duygu          : {DUYGU_TR.get(d, d)}  (%{hepsi['emotion'][d]:.1f})")
print("=" * 40)

plt.figure(figsize=(4.5, 4.5))
plt.imshow(resim); plt.axis("off")
plt.title(f"{hepsi['age']} yaş · {CINSIYET_TR.get(g, g)} · {DUYGU_TR.get(d, d)}",
          fontsize=13)
plt.show()


---
## 5 · Kendi fotoğrafınızı deneyin

Aşağıdaki hücreyi çalıştırıp bilgisayarınızdan bir görsel seçin. Fotoğraf
Colab oturumunuzda kalır, hiçbir yere gönderilmez; oturumu kapattığınızda silinir.


In [ ]:
from google.colab import files

yuklenen = files.upload()

for dosya_adi in yuklenen:
    kendi = Image.open(BytesIO(yuklenen[dosya_adi])).convert("RGB")
    kendi.save("kendi.jpg")

    try:
        s = analiz("kendi.jpg", ["age", "gender", "emotion"])
        g = s["dominant_gender"]; d = s["dominant_emotion"]
        plt.figure(figsize=(5, 5))
        plt.imshow(kendi); plt.axis("off")
        plt.title(f"{s['age']} yaş · {CINSIYET_TR.get(g, g)} · {DUYGU_TR.get(d, d)}", fontsize=13)
        plt.show()

        print("Duygu dağılımı:")
        for ad, p in sorted(s["emotion"].items(), key=lambda x: -x[1]):
            print(f"  %{p:5.1f}   {DUYGU_TR.get(ad, ad)}")
    except Exception as e:
        print("Analiz edilemedi:", e)


---
## 6 · Bonus — Yüz de bir vektör

Sunumda kelimelerin vektöre dönüştüğünü konuştuk: *kral − erkek + kadın ≈ kraliçe.*
Yüzler için de aynı şey geçerli.

`Facenet512` modeli bir yüzü **512 sayıdan oluşan bir vektöre** çeviriyor. Aynı
kişinin farklı fotoğrafları bu uzayda birbirine yakın, farklı kişiler uzak düşüyor.

Yüz tanıma sistemleri tam olarak böyle çalışır: karşılaştırma fotoğrafları arasında
değil, **vektörler arasında** yapılır.


In [ ]:
vektor = DeepFace.represent(
    img_path="foto.jpg", model_name="Facenet512",
    detector_backend=DETEKTOR, enforce_detection=False)[0]["embedding"]

vektor = np.array(vektor)
print("Vektör boyutu :", vektor.shape[0])
print("İlk 8 değer   :", np.round(vektor[:8], 4))

plt.figure(figsize=(11, 1.6))
plt.imshow(vektor.reshape(1, -1), aspect="auto", cmap="RdYlGn")
plt.yticks([]); plt.xlabel("512 boyut")
plt.title("Bu yüzün embedding vektörü")
plt.colorbar(); plt.tight_layout(); plt.show()


Şimdi iki farklı yüzü karşılaştıralım. Aynı fotoğrafın **parlaklığı değiştirilmiş**
hâli ile **bambaşka** bir yüzü aynı ölçüye vuruyoruz.


In [ ]:
from PIL import ImageEnhance

# aynı kişi, farklı aydınlatma
ImageEnhance.Brightness(resim).enhance(0.6).save("ayni_kisi.jpg")

# başka bir üretilmiş yüz
try:
    yuz_indir().save("baska_kisi.jpg")
    baska_var = True
except Exception:
    print("İkinci yüz indirilemedi, bu bölüm atlanıyor.")
    baska_var = False


def karsilastir(a, b, baslik):
    s = DeepFace.verify(a, b, model_name="Facenet512",
                        detector_backend=DETEKTOR, enforce_detection=False)
    karar = "AYNI KİŞİ" if s["verified"] else "FARKLI KİŞİ"
    print(f"{baslik:<34} mesafe={s['distance']:.3f}  (eşik {s['threshold']:.3f})  →  {karar}")


karsilastir("foto.jpg", "ayni_kisi.jpg", "Aynı yüz, farklı aydınlatma")
if baska_var:
    karsilastir("foto.jpg", "baska_kisi.jpg", "Farklı iki yüz")


> **Konuşma notu:** Aydınlatma değişti, pikseller tamamen farklı — ama vektörler
> hâlâ yakın. Model pikselleri değil, **kimliği** kodlamayı öğrenmiş. Telefonunuzun
> yüz tanıma özelliği de aynı prensiple çalışıyor.


---
## 7 · Sınırlar ve sorumluluk

Bu demoyu kapatmadan önce söylenmesi gereken birkaç şey var:

- **Yaş tahmini yaklaşıktır.** Model, eğitildiği veri setindeki insanlara benziyorsanız
  iyi çalışır; benzemiyorsanız ciddi şekilde şaşar.
- **Cinsiyet ikili değildir.** Model iki sınıfa zorlanmış durumda; bu bir gerçeklik
  değil, veri setinin kısıtı.
- **Duygu, yüz ifadesinden okunmaz.** Model yüz kaslarının konumunu sınıflandırıyor;
  insanın ne hissettiğini değil. Gülümseyen biri mutlu olmayabilir.
- **Yanlılık gerçektir.** Yüz veri setleri tarihsel olarak belirli etnik gruplara
  ağırlık verdi; doğruluk gruplara göre belirgin şekilde değişiyor.
- **Biyometrik veri hassas veridir.** KVKK ve GDPR kapsamında özel nitelikli kişisel
  veri sayılır. Bu defterde fotoğraf Colab oturumunda kalıyor, hiçbir yere gitmiyor.

> **Konuşma notu:** Bu bölümü atlamayın. Bir modelin çalışıyor olması, o modeli
> kullanmanın doğru olduğu anlamına gelmiyor. Teknik yeterlilik ile etik yeterlilik
> ayrı şeyler.


---
## Özet

Bugün anlattığımız her parçayı çalışırken gördük:

| Adım | Arkasında ne var |
|---|---|
| Yüzü bulmak | CNN — filtreler görüntüde gezdi |
| Kırpıp ölçeklemek | Normalizasyon — 150.528 sayı |
| Duygu | 7 sınıflı softmax |
| Yaş | Regresyon — tek sayı |
| Cinsiyet | İkili sınıflandırma |
| Kimlik | 512 boyutlu embedding, vektörler arası mesafe |

Ve hepsinin ortak noktası: **hiçbir kuralı biz yazmadık.**
Bütün bu davranışlar veriden öğrenildi.

---

### Denemek isterseniz

- Aynı kişinin gülümseyen ve nötr iki fotoğrafını karşılaştırın — duygu değişiyor mu?
- Fotoğrafı gri tonlamaya çevirin, sonucu karşılaştırın
- `detector_backend` değerini `"retinaface"` yapın: daha yavaş ama daha isabetli
- Kalabalık bir fotoğraf verin — kaç yüz buluyor?

*Deep Learning Master Class · Kuantum Bootcamp · Veysel Murat Görken*

*Kaynak: [generative-ai-workshop](https://github.com/grknc/generative-ai-workshop)*
